# Online Retail Data Cleaning

In the previous notebook, I explored the dataset and identified several
types of unusual records.

In this notebook, I will clean the data based on those findings and create
a dataset that can be used for the analysis.

I will keep the original data unchanged and create separate cleaned data
for analysis.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

In [3]:
df.shape

(541909, 8)

In [4]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

transaction_type = pd.Series({
    "Positive Quantity": (df["Quantity"] > 0).sum(),
    "Negative Quantity": (df["Quantity"] < 0).sum(),
    "Cancellation Invoice": df["InvoiceNo"].str.startswith("C").sum(),
    "Negative Quantity + Cancellation": (
        (df["Quantity"] < 0) &
        (df["InvoiceNo"].str.startswith("C"))
    ).sum(),
    "Negative Quantity + Non-Cancellation": (
        (df["Quantity"] < 0) &
        (~df["InvoiceNo"].str.startswith("C"))
    ).sum()
})

transaction_type

Positive Quantity                       531285
Negative Quantity                        10624
Cancellation Invoice                      9288
Negative Quantity + Cancellation          9288
Negative Quantity + Non-Cancellation      1336
dtype: int64

In [5]:
negative_price = df[df["UnitPrice"] < 0]

negative_price.shape

(2, 8)

In [6]:
negative_price

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


 3.1 Negative Unit Prices

I found two transactions with a negative `UnitPrice`.

Both records have the description `Adjust bad debt` and `StockCode` `B`.
They also have no CustomerID.

These are accounting adjustment records rather than normal product sales.
Therefore, they should not be included when calculating product sales revenue.

I will keep them in the original dataset but exclude them from the
sales-analysis dataset.

In [7]:
df.nlargest(10, "Quantity")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
74614,542504,37413,NaN,5568,2011-01-28 12:03:00,0.00,NaN,United Kingdom
421632,573008,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,4800,2011-10-27 12:26:00,0.21,12901.0,United Kingdom
206121,554868,22197,SMALL POPCORN HOLDER,4300,2011-05-27 10:52:00,0.72,13135.0,United Kingdom
220843,556231,85123A,?,4000,2011-06-09 15:04:00,0.00,NaN,United Kingdom
97432,544612,22053,EMPIRE DESIGN ROSETTE,3906,2011-02-22 10:43:00,0.82,18087.0,United Kingdom
270885,560599,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,3186,2011-07-19 17:04:00,0.06,14609.0,United Kingdom
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom


 3.2 Unusually Large Quantities

I checked the transactions with the largest quantities to see whether they
were obvious data errors.

The largest quantities include valid-looking product transactions with a
CustomerID and a positive UnitPrice. Therefore, a large quantity by itself
is not enough to classify a transaction as invalid.

Some other high-quantity records have a zero UnitPrice or missing
Description, which needs to be considered separately.

I will not remove transactions based only on a quantity threshold. Instead,
the cleaning rules will consider the transaction type, quantity, price, and
other available information together.

In [8]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [9]:
df.nlargest(10, "Revenue")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.60
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,77183.60
222680,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,38970.00
15017,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,13541.33
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06
173382,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,8142.75
348325,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450.0,United Kingdom,7144.72
52711,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749.0,United Kingdom,6539.40
160546,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749.0,United Kingdom,6539.40
421601,573003,23084,RABBIT NIGHT LIGHT,2400,2011-10-27 12:11:00,2.08,14646.0,Netherlands,4992.00


In [10]:
df[df["Description"].str.contains(
    "fee|postage|bad debt",
    case=False,
    na=False
)][
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "Revenue"]
]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,Revenue
45,536370,POST,POSTAGE,3,18.00,54.00
53,536373,37370,RETRO COFFEE MUGS ASSORTED,6,1.06,6.36
70,536375,37370,RETRO COFFEE MUGS ASSORTED,6,1.06,6.36
283,536396,37370,RETRO COFFEE MUGS ASSORTED,6,1.06,6.36
386,536403,POST,POSTAGE,1,15.00,15.00
...,...,...,...,...,...,...
541407,581498,22304,COFFEE MUG BLUE PAISLEY DESIGN,1,4.96,4.96
541540,581498,DOT,DOTCOM POSTAGE,1,1714.17,1714.17
541730,581570,POST,POSTAGE,1,18.00,18.00
541767,581574,POST,POSTAGE,2,18.00,36.00


In [11]:
df[df["Description"].str.contains(
    "fee|postage|bad debt",
    case=False,
    na=False
)]["Description"].value_counts()

Description
POSTAGE                                1252
DOTCOM POSTAGE                          709
SET OF TEA COFFEE SUGAR TINS PANTRY     509
SET 3 RETROSPOT TEA,COFFEE,SUGAR        449
LONDON BUS COFFEE MUG                   328
QUEENS GUARD COFFEE MUG                 279
COFFEE MUG APPLES DESIGN                273
COFFEE MUG CAT + BIRD DESIGN            270
COFFEE MUG PEARS  DESIGN                217
COFFEE MUG DOG + BALL DESIGN            205
RETRO COFFEE MUGS ASSORTED              171
POLKADOT COFFEE CUP & SAUCER PINK       113
ENAMEL PINK COFFEE CONTAINER             77
BLUE POLKADOT COFFEE MUG                 76
BAKING MOULD TOFFEE CUP CHOCOLATE        73
WHITE TEA,COFFEE,SUGAR JARS              72
RED POLKADOT COFFEE  MUG                 63
COFFEE SCENT PILLAR CANDLE               63
COFFEE MUG BLUE PAISLEY DESIGN           62
BLACK TEA,COFFEE,SUGAR JARS              53
COFFEE MUG PINK PAISLEY DESIGN           52
KENSINGTON COFFEE SET                    34
AMAZON FEE          

In [12]:
non_product = df[
    df["Description"].isin([
        "POSTAGE",
        "DOTCOM POSTAGE",
        "AMAZON FEE",
        "Adjust bad debt"
    ])
]

non_product.shape

(1998, 9)

In [13]:
non_product["Description"].value_counts()

Description
POSTAGE            1252
DOTCOM POSTAGE      709
AMAZON FEE           34
Adjust bad debt       3
Name: count, dtype: int64

In [14]:
non_product[[
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
    "Revenue"
]]

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Revenue
45,536370,POST,POSTAGE,3,18.00,12583.0,54.00
386,536403,POST,POSTAGE,1,15.00,12791.0,15.00
1123,536527,POST,POSTAGE,1,18.00,12662.0,18.00
1814,536544,DOT,DOTCOM POSTAGE,1,569.77,NaN,569.77
3041,536592,DOT,DOTCOM POSTAGE,1,607.49,NaN,607.49
...,...,...,...,...,...,...,...
541216,581494,POST,POSTAGE,2,18.00,12518.0,36.00
541540,581498,DOT,DOTCOM POSTAGE,1,1714.17,NaN,1714.17
541730,581570,POST,POSTAGE,1,18.00,12662.0,18.00
541767,581574,POST,POSTAGE,2,18.00,12526.0,36.00


 3.3 Non-Product Transactions

I found 1,998 transactions that are not normal product sales.

They consist of:

- 1,252 `POSTAGE` transactions
- 709 `DOTCOM POSTAGE` transactions
- 34 `AMAZON FEE` transactions
- 3 `Adjust bad debt` transactions

These transactions can have positive revenue values, but they represent
shipping charges, fees, or accounting adjustments rather than products.

For the product sales analysis, I will exclude these records while keeping
them in the original dataset.

In [15]:
zero_price = df[df["UnitPrice"] == 0]

pd.crosstab(
    zero_price["Quantity"] > 0,
    zero_price["Quantity"] < 0
)

Quantity,False,True
Quantity,,
False,0,1336
True,1179,0


In [16]:
zero_price.groupby(
    zero_price["Quantity"] > 0
)[["Quantity"]].count()

,Quantity
Quantity,
False,1336
True,1179


### 3.4 Zero-Price Transactions

There are 2,515 transactions with a `UnitPrice` of zero.

The transactions split into:

- 1,179 positive-quantity transactions
- 1,336 negative-quantity transactions

The 1,336 negative-quantity records match the group of negative transactions
without a cancellation invoice prefix that I investigated earlier. Many of
these appear to be operational or stock-related records.

The 1,179 positive-quantity records have no revenue because their unit price
is zero. They will therefore be excluded from the product sales dataset.

I will keep all of these records in the original dataset.

In [17]:
sales_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

In [18]:
sales_df.shape

(530104, 9)

In [19]:
sales_df["Quantity"].min()

np.int64(1)

In [20]:
sales_df["UnitPrice"].min()

np.float64(0.001)

In [21]:
sales_df["Revenue"].min()

np.float64(0.001)

In [23]:
sales_df["Description"].isin([
    "POSTAGE",
    "DOTCOM POSTAGE",
    "AMAZON FEE",
    "Adjust bad debt"
]).sum()

np.int64(1835)

In [24]:
sales_df[
    sales_df["Description"].isin([
        "POSTAGE",
        "DOTCOM POSTAGE",
        "AMAZON FEE",
        "Adjust bad debt"
    ])
]["Description"].value_counts()

Description
POSTAGE            1126
DOTCOM POSTAGE      706
AMAZON FEE            2
Adjust bad debt       1
Name: count, dtype: int64

In [25]:
non_product_items = [
    "POSTAGE",
    "DOTCOM POSTAGE",
    "AMAZON FEE",
    "Adjust bad debt"
]

sales_df = sales_df[
    ~sales_df["Description"].isin(non_product_items)
].copy()

In [26]:
sales_df.shape

(528269, 9)

In [27]:
sales_df["Description"].isin(non_product_items).sum()

np.int64(0)

### 3.5 Creating the Sales Dataset

After investigating the unusual transactions, I created a separate
`sales_df` containing transactions that can be treated as product sales.

The current rules are:

- Quantity must be greater than 0.
- UnitPrice must be greater than 0.
- Postage, fees, and bad-debt adjustment records are excluded.
- The original `df` remains unchanged.

This dataset will be used for the main product sales and revenue analysis.

In [28]:
sales_df["CustomerID"].isna().sum()

np.int64(131500)

In [29]:
sales_df["CustomerID"].notna().sum()

np.int64(396769)

In [30]:
sales_df["CustomerID"].isna().mean() * 100

np.float64(24.89262099422832)

### 3.6 CustomerID in the Sales Dataset

After the initial cleaning, 396,769 sales transactions have a CustomerID,
while 131,500 transactions do not.

This means 24.89% of the sales transactions cannot be directly linked to a
customer.

I decided to keep these transactions in the sales dataset because they still
contain useful information for overall sales, product, country, and
time-based analysis.

For customer-level analysis, I will use only transactions where CustomerID
is available.

In [31]:
sales_df["Revenue"].equals(
    sales_df["Quantity"] * sales_df["UnitPrice"]
)

True

In [32]:
sales_df["Revenue"].sum()

np.float64(10357510.744)

In [33]:
sales_df["Revenue"].describe()

count    528269.000000
mean         19.606509
std         269.006989
min           0.001000
25%           3.750000
50%           9.900000
75%          17.500000
max      168469.600000
Name: Revenue, dtype: float64

### 3.7 Revenue Validation

I created the `Revenue` column using:

`Revenue = Quantity × UnitPrice`

I verified the calculation against the original columns, and the result was
correct for all rows.

The cleaned sales dataset currently contains 528,269 transactions with total
revenue of approximately 10.36 million.

The revenue distribution is highly skewed, with a small number of transactions
having much higher revenue than the typical transaction.

I will investigate these high-value transactions during exploratory analysis
rather than removing them during cleaning.

In [34]:
sales_df.duplicated().sum()

np.int64(5226)

In [35]:
sales_df[[
    "InvoiceNo",
    "StockCode",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "Country"
]].isna().sum()

InvoiceNo      0
StockCode      0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

In [36]:
(sales_df["Quantity"] < 0).sum()

np.int64(0)

In [37]:
(sales_df["UnitPrice"] <= 0).sum()

np.int64(0)

In [38]:
duplicates = sales_df[sales_df.duplicated(keep=False)]

duplicates.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,4.95
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2.10
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1.25
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1.25
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2.95
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2.10
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2.95
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,4.95
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom,2.95
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom,2.95


In [39]:
duplicates.shape

(10068, 9)

In [40]:
duplicates.sort_values(
    ["InvoiceNo", "StockCode", "Description"]
).head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1.25
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1.25
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,4.95
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,4.95
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2.10
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,2.10
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2.95
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom,2.95
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,3.30
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom,1.65


In [41]:
exact_duplicates = sales_df[sales_df.duplicated()]

exact_duplicates.shape

(5226, 9)

In [42]:
exact_duplicates["Revenue"].sum()

np.float64(24573.74)

In [43]:
exact_duplicates["Revenue"].describe()

count    5226.000000
mean        4.702208
std         8.624925
min         0.120000
25%         1.250000
50%         2.500000
75%         4.950000
max       244.800000
Name: Revenue, dtype: float64

In [44]:
before_duplicates = sales_df.shape[0]

sales_df = sales_df.drop_duplicates().copy()

after_duplicates = sales_df.shape[0]

before_duplicates, after_duplicates

(528269, 523043)

In [45]:
sales_df.duplicated().sum()

np.int64(0)

In [46]:
sales_df["Revenue"].sum()

np.float64(10332937.004)

### 3.8 Removing Exact Duplicate Records

The sales dataset contained 5,226 exact duplicate rows.

These records were identical across all available columns, including the
invoice, product, quantity, price, customer, and country.

Because exact duplicates can artificially increase transaction counts and
revenue, I removed them using `drop_duplicates()`.

After removal, the sales dataset contains 523,043 transactions and no
remaining exact duplicates.

The revenue decreased by £24,573.74, which matches the revenue contributed
by the duplicate records.

In [47]:
sales_df[[
    "StockCode",
    "Description",
    "CustomerID",
    "Country"
]].isna().sum()

StockCode           0
Description         0
CustomerID     131466
Country             0
dtype: int64

In [48]:
sales_df["Description"].value_counts().head(10)

Description
WHITE HANGING HEART T-LIGHT HOLDER    2311
JUMBO BAG RED RETROSPOT               2109
REGENCY CAKESTAND 3 TIER              2007
PARTY BUNTING                         1699
LUNCH BAG RED RETROSPOT               1581
ASSORTED COLOUR BIRD ORNAMENT         1476
SET OF 3 CAKE TINS PANTRY DESIGN      1392
PACK OF 72 RETROSPOT CAKE CASES       1352
LUNCH BAG  BLACK SKULL.               1301
NATURAL SLATE HEART CHALKBOARD        1255
Name: count, dtype: int64

In [49]:
sales_df["StockCode"].value_counts().head(10)

StockCode
85123A    2253
85099B    2109
22423     2007
47566     1699
20725     1582
84879     1476
22197     1418
22720     1392
21212     1352
22383     1306
Name: count, dtype: int64

In [50]:
product_mapping = (
    sales_df.groupby("StockCode")["Description"]
    .nunique()
    .sort_values(ascending=False)
)

product_mapping.head(20)

StockCode
23196     4
23236     4
23203     3
23231     3
23126     3
22937     3
23209     3
22776     3
23131     3
23240     3
23244     3
23366     3
23370     3
23396     3
23413     3
23535     3
17107D    3
21928     2
21112     2
22900     2
Name: Description, dtype: int64

In [51]:
problem_stockcodes = product_mapping[product_mapping > 1].index

sales_df[
    sales_df["StockCode"].isin(problem_stockcodes)
][["StockCode", "Description"]].drop_duplicates().head(30)

,StockCode,Description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
8,22632,HAND WARMER RED POLKA DOT
40,22900,SET 2 TEA TOWELS I LOVE LONDON
58,82486,WOOD S/3 CABINET ANT WHITE FINISH
90,84997B,RED 3 PIECE RETROSPOT CUTLERY SET
91,84997C,BLUE 3 PIECE POLKADOT CUTLERY SET
93,20725,LUNCH BAG RED RETROSPOT
119,21175,GIN + TONIC DIET METAL SIGN
138,22778,GLASS CLOCHE SMALL


In [52]:
sales_df[
    sales_df["StockCode"] == "23196"
][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description


In [53]:
sales_df[
    sales_df["StockCode"] == "23236"
][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description


In [54]:
sales_df["StockCode"].isin(["23196", "23236"]).value_counts()

StockCode
False    523043
Name: count, dtype: int64

In [55]:
sales_df["StockCode"].dtype

dtype('O')

In [56]:
product_mapping = (
    sales_df.groupby("StockCode")["Description"]
    .nunique()
    .sort_values(ascending=False)
)

product_mapping.head(20)

StockCode
23196     4
23236     4
23203     3
23231     3
23126     3
22937     3
23209     3
22776     3
23131     3
23240     3
23244     3
23366     3
23370     3
23396     3
23413     3
23535     3
17107D    3
21928     2
21112     2
22900     2
Name: Description, dtype: int64

In [57]:
product_mapping[product_mapping > 1].head(20)

StockCode
23196     4
23236     4
23203     3
23231     3
23126     3
22937     3
23209     3
22776     3
23131     3
23240     3
23244     3
23366     3
23370     3
23396     3
23413     3
23535     3
17107D    3
21928     2
21112     2
22900     2
Name: Description, dtype: int64

In [58]:
problem_stockcodes = product_mapping[product_mapping > 1].index

sales_df[
    sales_df["StockCode"].isin(problem_stockcodes)
][["StockCode", "Description"]].drop_duplicates().head(50)

,StockCode,Description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
8,22632,HAND WARMER RED POLKA DOT
40,22900,SET 2 TEA TOWELS I LOVE LONDON
58,82486,WOOD S/3 CABINET ANT WHITE FINISH
90,84997B,RED 3 PIECE RETROSPOT CUTLERY SET
91,84997C,BLUE 3 PIECE POLKADOT CUTLERY SET
93,20725,LUNCH BAG RED RETROSPOT
119,21175,GIN + TONIC DIET METAL SIGN
138,22778,GLASS CLOCHE SMALL


In [59]:
sales_df[
    sales_df["StockCode"].astype(str).str.strip() == "23196"
][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
237422,23196,RETRO LEAVES MAGNETIC NOTEPAD
238991,23196,RETO LEAVES MAGNETIC SHOPPING LIST
246802,23196,LEAVES MAGNETIC SHOPPING LIST
252851,23196,VINTAGE LEAF MAGNETIC NOTEPAD


In [60]:
sales_df[
    sales_df["StockCode"].astype(str).str.strip() == "23236"
][["StockCode", "Description"]].drop_duplicates()

,StockCode,Description
218408,23236,DOILEY STORAGE TIN
220496,23236,DOILEY BISCUIT TIN
290770,23236,STORAGE TIN VINTAGE DOILEY
292790,23236,STORAGE TIN VINTAGE DOILY


### 3.9 StockCode and Description Consistency

I checked whether each StockCode maps to a single product description.

Some StockCodes are associated with multiple descriptions in the source data.
For example, StockCode `23196` appears with several descriptions related to
magnetic notepads and shopping lists, while StockCode `23236` appears with
different descriptions for storage/biscuit tins.

These differences may reflect changes or inconsistencies in the original
product records.

I will not overwrite or remove these descriptions because doing so would
introduce assumptions into the source data.

For product-level analysis, StockCode will be treated as the product
identifier, while Description will be retained as the descriptive field.

In [61]:
description_mapping = (
    sales_df.groupby("Description")["StockCode"]
    .nunique()
    .sort_values(ascending=False)
)

description_mapping.head(20)

Description
METAL SIGN,CUPCAKE SINGLE HOOK         6
CHILDRENS CUTLERY POLKADOT BLUE        2
CHILDRENS CUTLERY POLKADOT GREEN       2
CHARLIE + LOLA BISCUITS TINS           2
3 WHITE CHOC MORRIS BOXED CANDLES      2
75 GREEN FAIRY CAKE CASES              2
75 GREEN PETIT FOUR CASES              2
BLACK CHUNKY BEAD BRACELET W STRAP     2
CHILDRENS CUTLERY POLKADOT PINK        2
CHILDRENS CUTLERY RETROSPOT RED        2
CHARLIE AND LOLA FIGURES TINS          2
CHARLIE AND LOLA TABLE TINS            2
CHARLIE LOLA BLUE HOT WATER BOTTLE     2
EAU DE NILE JEWELLED PHOTOFRAME        2
BLACK DROP EARRINGS W LONG BEADS       2
3D DOG PICTURE PLAYING CARDS           2
FRENCH FLORAL CUSHION COVER            2
3D SHEET OF CAT STICKERS               2
3D SHEET OF DOG STICKERS               2
CHARLIE+LOLA RED HOT WATER BOTTLE      2
Name: StockCode, dtype: int64

In [62]:
description_mapping[description_mapping > 1].head(20)

Description
METAL SIGN,CUPCAKE SINGLE HOOK         6
CHILDRENS CUTLERY POLKADOT BLUE        2
CHILDRENS CUTLERY POLKADOT GREEN       2
CHARLIE + LOLA BISCUITS TINS           2
3 WHITE CHOC MORRIS BOXED CANDLES      2
75 GREEN FAIRY CAKE CASES              2
75 GREEN PETIT FOUR CASES              2
BLACK CHUNKY BEAD BRACELET W STRAP     2
CHILDRENS CUTLERY POLKADOT PINK        2
CHILDRENS CUTLERY RETROSPOT RED        2
CHARLIE AND LOLA FIGURES TINS          2
CHARLIE AND LOLA TABLE TINS            2
CHARLIE LOLA BLUE HOT WATER BOTTLE     2
EAU DE NILE JEWELLED PHOTOFRAME        2
BLACK DROP EARRINGS W LONG BEADS       2
3D DOG PICTURE PLAYING CARDS           2
FRENCH FLORAL CUSHION COVER            2
3D SHEET OF CAT STICKERS               2
3D SHEET OF DOG STICKERS               2
CHARLIE+LOLA RED HOT WATER BOTTLE      2
Name: StockCode, dtype: int64

### 3.10 Description and StockCode Consistency

I also checked the reverse relationship between Description and StockCode.

Some descriptions are associated with multiple StockCodes, while some
StockCodes are associated with multiple descriptions.

This confirms that StockCode and Description do not form a strict one-to-one
mapping in the source data.

I will preserve both fields without forcing a mapping. StockCode will be used
as the primary product identifier for product-level analysis, while
Description will be retained for interpretation and reporting.

In [63]:
sales_df["CustomerID"].describe()

count    391577.00000
mean      15295.00817
std        1710.25360
min       12346.00000
25%       13969.00000
50%       15157.00000
75%       16794.00000
max       18287.00000
Name: CustomerID, dtype: float64

In [64]:
sales_df["CustomerID"].nunique()

4338

In [65]:
sales_df["CustomerID"].dropna().astype(int).head(20)

0     17850
1     17850
2     17850
3     17850
4     17850
5     17850
6     17850
7     17850
8     17850
9     13047
10    13047
11    13047
12    13047
13    13047
14    13047
15    13047
16    13047
17    13047
18    13047
19    13047
Name: CustomerID, dtype: int64

### 3.11 CustomerID Validation

The available CustomerIDs were checked after the cleaning process.

There are 391,577 sales transactions with a CustomerID, representing 4,338
unique customers. The IDs fall within a consistent numeric range and no
obvious invalid values such as negative or zero IDs were found.

Missing CustomerIDs will be retained in the sales dataset because these
transactions are still useful for overall sales, product, country, and
time-based analysis.

For customer-level analysis, only transactions with a known CustomerID will
be used.

In [66]:
print("Rows:", len(sales_df))
print("Columns:", sales_df.shape[1])
print("Missing critical fields:")
print(
    sales_df[
        ["InvoiceNo", "StockCode", "Description",
         "Quantity", "InvoiceDate", "UnitPrice", "Country"]
    ].isna().sum()
)

print("\nNegative quantities:", (sales_df["Quantity"] < 0).sum())
print("Zero/negative prices:", (sales_df["UnitPrice"] <= 0).sum())
print("Exact duplicates:", sales_df.duplicated().sum())
print("Missing CustomerID:", sales_df["CustomerID"].isna().sum())
print("Total revenue:", sales_df["Revenue"].sum())

Rows: 523043
Columns: 9
Missing critical fields:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

Negative quantities: 0
Zero/negative prices: 0
Exact duplicates: 0
Missing CustomerID: 131466
Total revenue: 10332937.004


## 4. Final Data Cleaning Summary

The original dataset contained 541,909 transaction records.

After investigating transaction types, pricing, non-product records, duplicate
rows, and missing values, I created a separate `sales_df` for product sales
analysis.

The cleaning rules were:

- Keep transactions with positive quantities.
- Keep transactions with positive unit prices.
- Exclude postage, fees, and bad-debt adjustment records from product sales.
- Remove exact duplicate records.
- Preserve transactions with missing CustomerID because they remain useful for
  overall sales analysis.
- Preserve the original `df` without modification.

The final sales dataset contains 523,043 transactions and total revenue of
approximately £10.33 million.

All critical analytical fields are complete, with no missing InvoiceNo,
StockCode, Description, Quantity, InvoiceDate, UnitPrice, or Country values.

The dataset is now ready for exploratory data analysis.